In [3]:
import pandas as pd
import numpy as np
import ast

from sklearn.feature_extraction.text import TfidfVectorizer

In [5]:
# load the processed dataframe from the repo structure
disneyland_df = pd.read_csv('../data/processed/disneyland_df_processed.csv')

# quick check
print("shape:", disneyland_df.shape)
print("\ncolumns:")
print(disneyland_df.columns.tolist())

shape: (42479, 10)

columns:
['Review_ID', 'Rating', 'Year_Month', 'Reviewer_Location', 'Review_Text', 'Branch', 'review_cleaned', 'clean_review_wo_stopwords', 'tokenized_texts', 'cleaned_texts']


In [7]:
# convert tokenized_texts back to lists if they were read in as strings
if isinstance(disneyland_df['tokenized_texts'].iloc[0], str):
    disneyland_df['tokenized_texts'] = disneyland_df['tokenized_texts'].apply(ast.literal_eval)

# verify
print(type(disneyland_df['tokenized_texts'].iloc[0]))

<class 'list'>


In [11]:
# basic checks before modeling
print(disneyland_df['Branch'].value_counts())
print("\nmissing cleaned_texts:", disneyland_df['cleaned_texts'].isna().sum())
print("missing tokenized_texts:", disneyland_df['tokenized_texts'].isna().sum())

# preview the main columns we will use
disneyland_df[['Branch', 'cleaned_texts', 'tokenized_texts']].head()

Branch
Disneyland_California    19287
Disneyland_Paris         13598
Disneyland_HongKong       9594
Name: count, dtype: int64

missing cleaned_texts: 0
missing tokenized_texts: 0


,Branch,cleaned_texts,tokenized_texts
0,Disneyland_HongKong,ever anywhere find similar layout walk main st...,"[ever, anywhere, find, similar, layout, walk, ..."
1,Disneyland_HongKong,since last visit yet stay tomorrowland aka mar...,"[since, last, visit, yet, stay, tomorrowland, ..."
2,Disneyland_HongKong,thanks god hot humid visit otherwise big issue...,"[thanks, god, hot, humid, visit, otherwise, bi..."
3,Disneyland_HongKong,great compact unfortunately bit maintenance wo...,"[great, compact, unfortunately, bit, maintenan..."
4,Disneyland_HongKong,location city around hour kowlon kid everythin...,"[location, city, around, hour, kowlon, kid, ev..."


In [15]:
# split the dataframe into the 3 Disneyland branches
california_reviews = disneyland_df[disneyland_df['Branch'] == 'Disneyland_California'].copy()
paris_reviews = disneyland_df[disneyland_df['Branch'] == 'Disneyland_Paris'].copy()
hong_kong_reviews = disneyland_df[disneyland_df['Branch'] == 'Disneyland_HongKong'].copy()

# reset index to keep each group clean and easier to work with
california_reviews.reset_index(drop=True, inplace=True)
paris_reviews.reset_index(drop=True, inplace=True)
hong_kong_reviews.reset_index(drop=True, inplace=True)

# check sizes
print("California shape:", california_reviews.shape)
print("Paris shape:", paris_reviews.shape)
print("Hong Kong shape:", hong_kong_reviews.shape)

California shape: (19287, 10)
Paris shape: (13598, 10)
Hong Kong shape: (9594, 10)


In [17]:
# quick preview of cleaned review text for each branch
print("California sample:")
print(california_reviews['cleaned_texts'].head(3))
print()

print("Paris sample:")
print(paris_reviews['cleaned_texts'].head(3))
print()

print("Hong Kong sample:")
print(hong_kong_reviews['cleaned_texts'].head(3))

California sample:
0    place always forever special feeling enter cha...
1    great simple fun thrill bring cash nothing che...
2    great crowd huge ride sometimes min wait worth...
Name: cleaned_texts, dtype: object

Paris sample:
0    hongkong far well look forward visit big enoug...
1    april easter weekend june choose date love mum...
2    fantastic place queue decent good year apparen...
Name: cleaned_texts, dtype: object

Hong Kong sample:
0    ever anywhere find similar layout walk main st...
1    since last visit yet stay tomorrowland aka mar...
2    thanks god hot humid visit otherwise big issue...
Name: cleaned_texts, dtype: object


In [19]:
# make sure there are no missing values in cleaned_texts
print("Missing cleaned_texts values:")
print("California:", california_reviews['cleaned_texts'].isna().sum())
print("Paris:", paris_reviews['cleaned_texts'].isna().sum())
print("Hong Kong:", hong_kong_reviews['cleaned_texts'].isna().sum())
print()

# check for blank strings after preprocessing
print("Blank cleaned_texts values:")
print("California:", (california_reviews['cleaned_texts'].str.strip() == "").sum())
print("Paris:", (paris_reviews['cleaned_texts'].str.strip() == "").sum())
print("Hong Kong:", (hong_kong_reviews['cleaned_texts'].str.strip() == "").sum())

Missing cleaned_texts values:
California: 0
Paris: 0
Hong Kong: 0

Blank cleaned_texts values:
California: 0
Paris: 0
Hong Kong: 0


In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer

def build_tfidf_matrix(text_series, min_df=5, max_df=0.80, ngram_range=(1, 1)):
    """
    Builds a TF-IDF matrix from cleaned review text.

    Parameters:
    text_series : pandas Series
        The cleaned review text.
    min_df : int
        Minimum number of documents a term must appear in.
    max_df : float
        Ignore terms that appear in more than this proportion of documents.
    ngram_range : tuple
        (1,1) = unigrams only, (1,2) = unigrams and bigrams.

    Returns:
    vectorizer : fitted TfidfVectorizer
    tfidf_matrix : sparse matrix
    feature_names : list of vocabulary terms
    """
    
    vectorizer = TfidfVectorizer(
        min_df=min_df,
        max_df=max_df,
        ngram_range=ngram_range
    )
    
    tfidf_matrix = vectorizer.fit_transform(text_series)
    feature_names = vectorizer.get_feature_names_out()
    
    return vectorizer, tfidf_matrix, feature_names

In [23]:
# build TF-IDF matrix for California reviews
california_vectorizer, california_tfidf, california_features = build_tfidf_matrix(
    california_reviews['cleaned_texts']
)

print("California TF-IDF matrix shape:", california_tfidf.shape)
print("California vocabulary size:", len(california_features))

California TF-IDF matrix shape: (19287, 5736)
California vocabulary size: 5736


In [25]:
# build TF-IDF matrix for Paris reviews
paris_vectorizer, paris_tfidf, paris_features = build_tfidf_matrix(
    paris_reviews['cleaned_texts']
)

print("Paris TF-IDF matrix shape:", paris_tfidf.shape)
print("Paris vocabulary size:", len(paris_features))

Paris TF-IDF matrix shape: (13598, 5930)
Paris vocabulary size: 5930


In [27]:
# build TF-IDF matrix for Hong Kong reviews
hong_kong_vectorizer, hong_kong_tfidf, hong_kong_features = build_tfidf_matrix(
    hong_kong_reviews['cleaned_texts']
)

print("Hong Kong TF-IDF matrix shape:", hong_kong_tfidf.shape)
print("Hong Kong vocabulary size:", len(hong_kong_features))

Hong Kong TF-IDF matrix shape: (9594, 3692)
Hong Kong vocabulary size: 3692


In [29]:
# organize the 3 branch datasets in one dictionary
# this makes it easier to loop through them later when we run NMF for multiple k values
group_data = {
    'California': {
        'df': california_reviews,
        'vectorizer': california_vectorizer,
        'tfidf_matrix': california_tfidf,
        'features': california_features
    },
    'Paris': {
        'df': paris_reviews,
        'vectorizer': paris_vectorizer,
        'tfidf_matrix': paris_tfidf,
        'features': paris_features
    },
    'Hong Kong': {
        'df': hong_kong_reviews,
        'vectorizer': hong_kong_vectorizer,
        'tfidf_matrix': hong_kong_tfidf,
        'features': hong_kong_features
    }
}

In [31]:
# quick summary so we can confirm all 3 matrices were created successfully
for group_name, group_info in group_data.items():
    print(group_name)
    print("Matrix shape:", group_info['tfidf_matrix'].shape)
    print("Vocabulary size:", len(group_info['features']))
    print()

California
Matrix shape: (19287, 5736)
Vocabulary size: 5736

Paris
Matrix shape: (13598, 5930)
Vocabulary size: 5930

Hong Kong
Matrix shape: (9594, 3692)
Vocabulary size: 3692



In [33]:
# preview a few vocabulary terms from each branch
print("California sample words:", california_features[:20])
print()
print("Paris sample words:", paris_features[:20])
print()
print("Hong Kong sample words:", hong_kong_features[:20])

California sample words: ['aaa' 'abandon' 'abc' 'abe' 'ability' 'abit' 'able' 'aboard' 'abominable'
 'abound' 'abraham' 'abrupt' 'abruptly' 'absence' 'absent' 'absolute'
 'absolutely' 'absolutley' 'absolutly' 'absorb']

Paris sample words: ['aback' 'abandon' 'ability' 'abit' 'able' 'abroad' 'abrupt' 'abruptly'
 'absence' 'absent' 'absoloutly' 'absolute' 'absolutely' 'absolutly'
 'absorb' 'absurd' 'abundance' 'abuse' 'abysmal' 'academy']

Hong Kong sample words: ['ability' 'abit' 'able' 'abound' 'abroad' 'abrupt' 'absolute'
 'absolutely' 'abundance' 'abundant' 'academy' 'accept' 'acceptable'
 'accesible' 'access' 'accessibility' 'accessible' 'accessory' 'accident'
 'accidentally']


### NMF Topic Modeling
This section tests the NMF topic model for each Disneyland location using k = 5, 8, 10, 12, and 15 (per instructions). 

For each model, we will: 
- fit NMF using each TFIDF matrix
- plot coherence and reconstruction error
- save top words for each topic

In [35]:
from sklearn.decomposition import NMF
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

In [37]:
# create separate tokenized text lists for each branch
california_tokens = california_reviews['tokenized_texts'].tolist()
paris_tokens = paris_reviews['tokenized_texts'].tolist()
hong_kong_tokens = hong_kong_reviews['tokenized_texts'].tolist()

# create gensim dictionaries for each branch
california_dictionary = Dictionary(california_tokens)
paris_dictionary = Dictionary(paris_tokens)
hong_kong_dictionary = Dictionary(hong_kong_tokens)

In [39]:
# add tokenized texts and dictionaries into group_data
group_data['California']['tokens'] = california_tokens
group_data['California']['dictionary'] = california_dictionary

group_data['Paris']['tokens'] = paris_tokens
group_data['Paris']['dictionary'] = paris_dictionary

group_data['Hong Kong']['tokens'] = hong_kong_tokens
group_data['Hong Kong']['dictionary'] = hong_kong_dictionary

In [41]:
def get_top_words_per_topic(model, feature_names, top_n=10):
    """
    Returns the top words for each topic in an NMF model.
    """
    topic_words = []
    
    for topic_idx, topic in enumerate(model.components_):
        top_word_indices = topic.argsort()[::-1][:top_n]
        top_words = [feature_names[i] for i in top_word_indices]
        topic_words.append(top_words)
    
    return topic_words

In [43]:
def compute_nmf_coherence(topic_words, tokenized_texts, dictionary):
    """
    Computes c_v coherence for a set of topic word lists.
    """
    coherence_model = CoherenceModel(
        topics=topic_words,
        texts=tokenized_texts,
        dictionary=dictionary,
        coherence='c_v'
    )
    
    return coherence_model.get_coherence()

In [45]:
def evaluate_nmf_models(tfidf_matrix, feature_names, tokenized_texts, dictionary, k_values):
    """
    Fits NMF models for multiple k values and stores the results.
    
    Returns a list of dictionaries with:
    - k
    - fitted model
    - reconstruction error
    - coherence score
    - top words per topic
    """
    
    results = []
    
    for k in k_values:
        print(f"Running NMF for k = {k}...")
        
        # fit the model
        nmf_model = NMF(
            n_components=k,
            random_state=42,
            init='nndsvda',
            max_iter=200
        )
        
        W = nmf_model.fit_transform(tfidf_matrix)
        H = nmf_model.components_
        
        # get top words for each topic
        topic_words = get_top_words_per_topic(nmf_model, feature_names, top_n=10)
        
        # compute coherence
        coherence_score = compute_nmf_coherence(
            topic_words=topic_words,
            tokenized_texts=tokenized_texts,
            dictionary=dictionary
        )
        
        # save outputs
        result = {
            'k': k,
            'model': nmf_model,
            'document_topic_matrix': W,
            'topic_term_matrix': H,
            'reconstruction_error': nmf_model.reconstruction_err_,
            'coherence_score': coherence_score,
            'topic_words': topic_words
        }
        
        results.append(result)
    
    return results

In [47]:
# topic counts
k_values = [5, 8, 10, 12, 15]
print(k_values)

[5, 8, 10, 12, 15]


In [49]:
# run NMF models for California
california_nmf_results = evaluate_nmf_models(
    tfidf_matrix=group_data['California']['tfidf_matrix'],
    feature_names=group_data['California']['features'],
    tokenized_texts=group_data['California']['tokens'],
    dictionary=group_data['California']['dictionary'],
    k_values=k_values
)

Running NMF for k = 5...


/opt/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


Running NMF for k = 8...
Running NMF for k = 10...


/opt/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


Running NMF for k = 12...
Running NMF for k = 15...


In [50]:
# run NMF models for Paris
paris_nmf_results = evaluate_nmf_models(
    tfidf_matrix=group_data['Paris']['tfidf_matrix'],
    feature_names=group_data['Paris']['features'],
    tokenized_texts=group_data['Paris']['tokens'],
    dictionary=group_data['Paris']['dictionary'],
    k_values=k_values
)

Running NMF for k = 5...
Running NMF for k = 8...
Running NMF for k = 10...
Running NMF for k = 12...


/opt/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


Running NMF for k = 15...


In [51]:
# run NMF models for Hong Kong
hong_kong_nmf_results = evaluate_nmf_models(
    tfidf_matrix=group_data['Hong Kong']['tfidf_matrix'],
    feature_names=group_data['Hong Kong']['features'],
    tokenized_texts=group_data['Hong Kong']['tokens'],
    dictionary=group_data['Hong Kong']['dictionary'],
    k_values=k_values
)

Running NMF for k = 5...
Running NMF for k = 8...
Running NMF for k = 10...
Running NMF for k = 12...
Running NMF for k = 15...
